In [1]:
# robust_ingest_with_pdfium_and_easyocr.py
# Paste into one notebook cell (or adapt to .py). Adjust CONFIG at top.

import os, re, json, math, glob, time, traceback
from pathlib import Path
import pdfplumber
import pytesseract
import cv2
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
import faiss
import csv


C:\ProgramData\anaconda3\envs\training_env\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
# optional: pypdfium2 for robust PDF -> image rendering
try:
    import pypdfium2 as pdfium
    HAS_PDFIUM = True
except Exception:
    HAS_PDFIUM = False

    

In [3]:
import os
import sys
import subprocess

def check_and_install_pytorch():
    """Check if correct PyTorch is installed, install if not"""
    try:
        import torch
        if not torch.cuda.is_available():
            print("Installing PyTorch with CUDA support...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", 
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"
            ])
            print("Please restart the script after installation.")
            return False
        return True
    except ImportError:
        print("PyTorch not found. Installing...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "torch", "torchvision", "torchaudio",
            "--index-url", "https://download.pytorch.org/whl/cu121"
        ])
        return True

# Check PyTorch installation
if check_and_install_pytorch():
    import torch
    print(f"\n{'='*50}")
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"{'='*50}\n")
    
    # Now initialize EasyOCR
    try:
        import easyocr
        # Force GPU usage
        easyocr_reader = easyocr.Reader(['en','id'], gpu=True)
        print("✓ EasyOCR initialized with GPU acceleration")
        
        # Test with a simple image
        test_text = easyocr_reader.readtext('Surat-Penawaran-Proyek.png', detail=0)
        print(f"Test OCR result: {test_text}")
        
    except Exception as e:
        print(f"EasyOCR error: {e}")


PyTorch Version: 2.8.0+cu128
CUDA Available: True
CUDA Version: 12.8
Device: NVIDIA GeForce RTX 4060 Laptop GPU
Memory: 8.6 GB

✓ EasyOCR initialized with GPU acceleration
Test OCR result: ['SURAT PENAWARAN PROYEK', 'Perusohoan Konstruksi Nuso', 'Contructior', 'Targgot [Tanggal]', 'Kepoda Yth ,', 'Bapak Ibu Noro Penerimo]', 'Dengon hormot', 'Bersomg', 'Suro', '<ami', 'Perusoragr', 'Korstuksi', 'Nusg', 'Contruction', 'Yong berlokosi', 'Jalan', 'Parbongunan', 'Jakorto', 'ingin', 'mengojukon', 'Denowaron', 'keriosomg', 'dalam', 'Dalo<songan Droye<', '[Narno] yong berlokosi', '[Alamat]', 'Perusghoar', 'Yodstuks', 'Nusc', 'Contruction meriliki pengolamon lebih dari [x] tohur', 'dalamn', 'bidang', 'konstruksi Yong mencokup berbogor', 'Proyek seperti [sebutkon jenis proyek, misolnyo', 'Perbangunan gedung', 'jalan;', 'embatan', 'Kamni', 'meriliki tim yang terdiri dari ohli yong', 'berpengolaman', 'berkompeten dalam', 'bidorgnyo mosing', 'mosing serto', 'didukung oleh', 'Perolotan vong', 'meme

In [4]:
# optional: EasyOCR fallback
try:
    import easyocr
    import torch
    HAS_EASYOCR = True
    
    # Check what's available
    if torch.cuda.is_available():
        # NVIDIA CUDA GPU available
        easyocr_reader = easyocr.Reader(['en','id'], gpu=True)
        print("EasyOCR: Using CUDA GPU acceleration")
    elif torch.backends.mps.is_available():
        # Apple Silicon MPS available
        easyocr_reader = easyocr.Reader(['en','id'], gpu=True)
        print("EasyOCR: Using Apple Silicon GPU acceleration")
    else:
        # CPU only
        easyocr_reader = easyocr.Reader(['en','id'], gpu=False)
        print("EasyOCR: Using CPU (GPU not available)")
except Exception as e:
    HAS_EASYOCR = False
    print(f"EasyOCR not available: {e}")

EasyOCR: Using CUDA GPU acceleration


In [5]:
# -------------------- CONFIG --------------------
LM_BASE_URL = os.environ.get("LM_BASE_URL", "http://localhost:1234/v1")
LM_API_KEY = os.environ.get("LM_API_KEY", "lm-studio")
LM_MODEL = os.environ.get("LM_MODEL", "google/gemma-3-27b")

TESSERACT_CMD = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # set to your tesseract path, or None
if TESSERACT_CMD:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD
MAX_TOKENS = 4096
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM = 384
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 6
TEMPERATURE = 0.0
VECTOR_INDEX_FILE = "rag_index.faiss"
METADATA_FILE = "rag_metadata.json"
DEBUG_OCR_DIR = "ocr_debug"
# -----------------------------------------------

In [6]:
# helper sanitize
def sanitize_basename(path):
    p = Path(path)
    base = p.stem.lower()
    base = re.sub(r'[^a-z0-9_]+', '_', base)
    base = re.sub(r'_{2,}', '_', base).strip('_')
    return base or "doc"


In [7]:
# -------------------- Image preprocessing & OCR utils --------------------
def load_image_cv2(path):
    arr = np.fromfile(path, dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None:
        img = cv2.cvtColor(np.array(Image.open(path)), cv2.COLOR_RGB2BGR)
    return img

def resize_max(img, max_side=2600, min_side=900):
    h, w = img.shape[:2]
    max_current = max(h, w)
    if max_current > max_side:
        scale = max_side / max_current
        return cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_CUBIC)
    if max_current < min_side:
        scale = min_side / max_current
        return cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_CUBIC)
    return img

def to_grayscale(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def denoise(img_gray):
    return cv2.fastNlMeansDenoising(img_gray, h=10)

def adaptive_thresh(img_gray):
    return cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)

def otsu_thresh(img_gray):
    _, th = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return th

def unsharp_mask(img_gray):
    blur = cv2.GaussianBlur(img_gray, (0,0), sigmaX=3)
    return cv2.addWeighted(img_gray, 1.5, blur, -0.5, 0)

def deskew(img_gray):
    coords = np.column_stack(np.where(img_gray < 255))
    if coords.size == 0:
        return img_gray
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    (h, w) = img_gray.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(img_gray, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return rotated

def ocr_with_confidence(pil_img, config="--oem 3 --psm 3"):
    try:
        data = pytesseract.image_to_data(pil_img, config=config, output_type=pytesseract.Output.DICT)
    except Exception:
        return "", 0.0, 0
    texts = []
    confs = []
    for txt, conf in zip(data.get('text', []), data.get('conf', [])):
        if txt and str(txt).strip():
            try:
                ic = int(float(conf))
            except Exception:
                ic = 0
            if ic > 0:
                texts.append(txt)
                confs.append(ic)
            else:
                texts.append(txt)
    joined = " ".join([t for t in texts]).strip()
    avg_conf = (sum(confs)/len(confs)) if confs else 0.0
    return joined, avg_conf, len(joined)

def try_preprocess_and_ocr_image(path, debug_dir=DEBUG_OCR_DIR):
    os.makedirs(debug_dir, exist_ok=True)
    img = load_image_cv2(path)
    img = resize_max(img)
    gray = to_grayscale(img)

    variants = []
    variants.append(("gray", gray))
    try: variants.append(("deskew", deskew(gray)))
    except: pass
    try: variants.append(("denoise", denoise(gray)))
    except: pass
    try: variants.append(("otsu", otsu_thresh(gray)))
    except: pass
    try: variants.append(("adaptive", adaptive_thresh(gray)))
    except: pass
    try: variants.append(("unsharp", unsharp_mask(gray)))
    except: pass

    morphs = []
    for lab, v in variants:
        morphs.append((lab + "_none", v))
        try:
            k = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
            morphs.append((lab + "_open", cv2.morphologyEx(v, cv2.MORPH_OPEN, k)))
            morphs.append((lab + "_close", cv2.morphologyEx(v, cv2.MORPH_CLOSE, k)))
        except Exception:
            pass

    configs = ["--oem 3 --psm 6", "--oem 3 --psm 3", "--oem 3 --psm 11", "--oem 1 --psm 6"]
    candidates = []
    for lab, proc in morphs:
        try:
            pil_img = Image.fromarray(proc) if len(proc.shape)==2 else Image.fromarray(cv2.cvtColor(proc, cv2.COLOR_BGR2RGB))
        except Exception:
            pil_img = Image.fromarray(proc)
        for cfg in configs:
            text, avg_conf, total_chars = ocr_with_confidence(pil_img, config=cfg)
            candidates.append({"variant": lab, "config": cfg, "text": text, "avg_conf": avg_conf, "total_chars": total_chars})

    # pick best by (total_chars, avg_conf)
    best = max(candidates, key=lambda c: (c["total_chars"], c["avg_conf"]))
    # save debug
    try:
        for lab, proc in morphs:
            if lab == best["variant"]:
                out_img = os.path.join(debug_dir, f"debug_best_{Path(path).stem}.png")
                cv2.imwrite(out_img, proc if proc.dtype==np.uint8 else proc.astype(np.uint8))
                break
        with open(os.path.join(debug_dir, f"debug_candidates_{Path(path).stem}.json"), "w", encoding="utf-8") as fh:
            json.dump(candidates, fh, indent=2, ensure_ascii=False)
    except Exception:
        pass

    # If tesseract produced something short AND easyocr exists, try easyocr
    if (not best["text"] or len(best["text"]) < 30) and HAS_EASYOCR:
        try:
            e_res = easyocr_reader.readtext(path, detail=0)
            e_text = " ".join(e_res)
            # compare lengths
            if len(e_text) > len(best["text"]):
                best = {"variant":"easyocr", "config":"easyocr", "text": e_text, "avg_conf":0, "total_chars":len(e_text)}
        except Exception:
            pass

    return best["text"], best


In [8]:
# -------------------- PDF -> text (robust) --------------------
def pdf_to_text(path, ocr_images=False, lang='ind+eng'):
    """
    Try:
     1) pdfplumber.extract_text() (fast, for born-digital PDFs)
     2) if empty -> render pages to images (pypdfium2 if available; otherwise pdfplumber.page.to_image)
     3) run try_preprocess_and_ocr_image on each page image and concat results
    """
    text_pages = []
    try:
        with pdfplumber.open(path) as pdf:
            for p in pdf.pages:
                page_text = p.extract_text()
                if page_text and page_text.strip():
                    text_pages.append(page_text)
                else:
                    # if requested, do per-page OCR via PIL image
                    if ocr_images:
                        try:
                            pil = p.to_image(resolution=300).original
                            page_text = pytesseract.image_to_string(pil, lang=lang)
                            if page_text and page_text.strip():
                                text_pages.append(page_text)
                                continue
                        except Exception:
                            pass
                    # leave placeholder, later we'll render via pdfium if available
                    text_pages.append("")  # placeholder for page i
    except Exception:
        # if pdfplumber fails entirely, proceed to rendering fallback
        text_pages = []

    # If any page is empty, try rendering pages with pdfium (preferred) or fallback to page.to_image
    need_render = any(not t or t.strip()=="" for t in text_pages) or len(text_pages)==0
    if not need_render:
        return "\n\n".join([t for t in text_pages if t and t.strip()])

    pages_text = []
    # Render with pypdfium2 if available
    if HAS_PDFIUM:
        try:
            pdf = pdfium.PdfDocument(path)
            for i in range(len(pdf)):
                # render as image at 150-300 DPI
                try:
                    pil = pdf.render_topil(i, scale=2.0)  # scale 2.0 approximates higher DPI
                    tmp_path = os.path.join(DEBUG_OCR_DIR, f"pdfpage_{Path(path).stem}_p{i}.png")
                    os.makedirs(DEBUG_OCR_DIR, exist_ok=True)
                    pil.save(tmp_path)
                    t, best = try_preprocess_and_ocr_image(tmp_path, debug_dir=DEBUG_OCR_DIR)
                    pages_text.append(t or "")
                except Exception as e:
                    pages_text.append("")
            pdf.close()
            return "\n\n".join([p for p in pages_text if p])
        except Exception:
            # fallback to pdfplumber page images
            pass

    # fallback: try pdfplumber page->image conversion and OCR per page
    try:
        with pdfplumber.open(path) as pdf:
            for i, p in enumerate(pdf.pages):
                try:
                    img = p.to_image(resolution=200).original
                    temp_path = os.path.join(DEBUG_OCR_DIR, f"pdfpage_{Path(path).stem}_fallback_p{i}.png")
                    os.makedirs(DEBUG_OCR_DIR, exist_ok=True)
                    img.save(temp_path)
                    t, best = try_preprocess_and_ocr_image(temp_path, debug_dir=DEBUG_OCR_DIR)
                    pages_text.append(t or "")
                except Exception:
                    pages_text.append("")
    except Exception:
        pass

    return "\n\n".join([p for p in pages_text if p])

In [9]:
# -------------------- image_to_text wrapper --------------------
def image_to_text(path, lang='ind+eng'):
    # quick direct attempt with tesseract
    try:
        text = pytesseract.image_to_string(Image.open(path), lang=lang)
        if text and text.strip():
            return text
    except Exception:
        pass
    # else fallback to preprocessing heavy function
    t, best = try_preprocess_and_ocr_image(path, debug_dir=DEBUG_OCR_DIR)
    return t

In [10]:
# -------------------- text cleaning + chunking --------------------
def clean_text(text):
    if not text:
        return ""
    t = re.sub(r'\r\n', '\n', text)
    t = re.sub(r'\n{3,}', '\n\n', t)
    t = re.sub(r'[ \t]+', ' ', t)
    return t.strip()

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = clean_text(text)
    if not text:
        return []
    n = len(text); chunks=[]; start=0; cid=0
    while start < n:
        end = start + chunk_size
        chunk_txt = text[start:end]
        chunks.append({"id": cid, "text": chunk_txt, "start": start, "end": min(end, n)})
        cid += 1
        if end >= n: break
        start = end - overlap
    return chunks

In [11]:
# -------------------- embeddings & faiss --------------------
print("Loading embedding model:", EMBED_MODEL_NAME)
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

def create_faiss_index(dim=EMBED_DIM):
    return faiss.IndexFlatIP(dim)

def save_index(index, filepath):
    faiss.write_index(index, filepath)

def load_index(filepath):
    return faiss.read_index(filepath)

def embed_texts(texts, batch_size=32):
    embs = embed_model.encode(texts, convert_to_numpy=True, show_progress_bar=False, batch_size=batch_size)
    faiss.normalize_L2(embs)
    return embs

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


In [12]:
# -------------------- ingest functions (single and folder) --------------------
def ingest_single_file(path, index_path=None, metadata_path=None, ocr_for_pdf=True):
    p = Path(path)
    if not p.exists(): raise FileNotFoundError(f"{path} not found")
    base = sanitize_basename(path)
    if index_path is None: index_path = f"rag_index_{base}.faiss"
    if metadata_path is None: metadata_path = f"rag_meta_{base}.json"

    ext = p.suffix.lower()
    if ext == ".pdf":
        raw = pdf_to_text(str(p), ocr_images=ocr_for_pdf)
    elif ext in [".png",".jpg",".jpeg",".tif",".tiff"]:
        raw = image_to_text(str(p))
    elif ext == ".txt":
        raw = p.read_text(encoding="utf-8", errors="ignore")
    else:
        raise ValueError("Unsupported file type")

    raw = clean_text(raw)
    if not raw:
        raise RuntimeError("No text extracted. Check ocr_debug/ for diagnostic images or enable EasyOCR as fallback.")

    chunks = chunk_text(raw)
    if not chunks:
        raise RuntimeError("Chunking produced no chunks (empty text).")
    all_chunks=[]; metadata=[]
    doc_id = base
    for c in chunks:
        uid = f"{doc_id}_c{c['id']}"
        all_chunks.append((uid, c["text"]))
        metadata.append({"uid":uid,"doc_id":doc_id,"source":str(p),"start":c["start"],"end":c["end"],"text":c["text"],"text_preview":c["text"][:400]})

    texts = [t for (_,t) in all_chunks]
    embs = embed_texts(texts)
    index = create_faiss_index(dim=embs.shape[1])
    index.add(embs)

    save_index(index, index_path)
    with open(metadata_path,"w",encoding="utf-8") as fh:
        json.dump(metadata, fh, indent=2, ensure_ascii=False)

    print(f"Ingested {len(all_chunks)} chunks from {path}")
    print(f"Saved index -> {index_path}")
    print(f"Saved metadata -> {metadata_path}")
    return index, metadata, index_path, metadata_path

def ingest_folder(folder, index_path=VECTOR_INDEX_FILE, metadata_path=METADATA_FILE, ocr_for_pdf=True):
    patterns = [os.path.join(folder, ext) for ext in ["**/*.pdf","**/*.png","**/*.jpg","**/*.jpeg","**/*.tif","**/*.tiff","**/*.txt"]]
    files=[]
    for pat in patterns: files.extend(glob.glob(pat, recursive=True))
    all_chunks=[]; metadata=[]; doc_counter=0
    for path in tqdm(sorted(set(files)), desc="Files"):
        ext = os.path.splitext(path)[1].lower()
        if ext == ".pdf":
            raw = pdf_to_text(path, ocr_images=ocr_for_pdf)
        elif ext in [".png",".jpg",".jpeg",".tif",".tiff"]:
            raw = image_to_text(path)
        elif ext == ".txt":
            with open(path,"r",encoding="utf-8",errors="ignore") as fh: raw = fh.read()
        else:
            raw=""
        raw = clean_text(raw)
        if not raw: continue
        doc_id = f"doc_{doc_counter}"; doc_counter += 1
        chunks = chunk_text(raw)
        for c in chunks:
            uid = f"{doc_id}_c{c['id']}"
            all_chunks.append((uid, c["text"]))
            metadata.append({"uid":uid,"doc_id":doc_id,"source":path,"start":c["start"],"end":c["end"],"text":c["text"],"text_preview":c["text"][:200]})
    if not all_chunks:
        print("No text chunks to ingest.")
        return None, None
    texts=[t for (_,t) in all_chunks]
    embs = embed_texts(texts)
    index = create_faiss_index(dim=embs.shape[1]); index.add(embs)
    with open(metadata_path,"w",encoding="utf-8") as fh: json.dump(metadata, fh, indent=2, ensure_ascii=False)
    save_index(index, index_path)
    print(f"Ingested {len(all_chunks)} chunks from {len(metadata)} source chunks. Index saved to {index_path}")
    return index, metadata

def ingest_path(path, index_dir=".", reuse=True, ocr_for_pdf=True):
    p = Path(path)
    if not p.exists(): raise FileNotFoundError(f"{path} not found")
    if p.is_file():
        base = sanitize_basename(path)
        index_path = os.path.join(index_dir, f"rag_index_{base}.faiss")
        metadata_path = os.path.join(index_dir, f"rag_meta_{base}.json")
        if reuse and os.path.exists(index_path) and os.path.exists(metadata_path):
            idx = load_index(index_path)
            with open(metadata_path,"r",encoding="utf-8") as fh: md = json.load(fh)
            return idx, md, index_path, metadata_path
        return ingest_single_file(path, index_path=index_path, metadata_path=metadata_path, ocr_for_pdf=ocr_for_pdf)
    else:
        idx, md = ingest_folder(path, index_path=os.path.join(index_dir, VECTOR_INDEX_FILE), metadata_path=os.path.join(index_dir, METADATA_FILE), ocr_for_pdf=ocr_for_pdf)
        return idx, md, os.path.join(index_dir, VECTOR_INDEX_FILE), os.path.join(index_dir, METADATA_FILE)

# -------------------- (remaining pipeline: embedding, retrieval, LM) --------------------
# You can reuse your existing extract_from_all_documents(), call_gemma_extract_rag(), etc.
# For brevity they are not repeated here; use the versions you already have that accept in-memory index+metadata.


In [13]:
# ---------- CELL 6: retrieval helpers ----------
def load_vectorstore(index_path=VECTOR_INDEX_FILE, metadata_path=METADATA_FILE):
    if not os.path.exists(index_path) or not os.path.exists(metadata_path):
        raise FileNotFoundError("Index or metadata file not found. Run ingest first.")
    index = load_index(index_path)
    with open(metadata_path, "r", encoding="utf-8") as fh:
        metadata = json.load(fh)
    # Ensure text present
    for m in metadata:
        if 'text' not in m:
            m['text'] = m.get('text_preview', '')
    return index, metadata

def retrieve(query, index, metadata, top_k=TOP_K):
    q_emb = embed_texts([query])[0:1]
    D, I = index.search(q_emb, top_k)
    results = []
    for idx in I[0]:
        if idx < 0 or idx >= len(metadata):
            continue
        results.append(metadata[idx])
    return results



In [14]:
# ---------- CELL 7: LM call + JSON extraction (robust) ----------
def extract_json_block(text):
    if not text or "{" not in text:
        return None
    start_idx = None
    depth = 0
    in_string = False
    escape = False
    for i, ch in enumerate(text):
        if start_idx is None:
            if ch == "{":
                start_idx = i
                depth = 1
                in_string = False
                escape = False
            else:
                continue
        else:
            if escape:
                escape = False
                continue
            if ch == "\\":
                escape = True
                continue
            if ch == '"' or ch == "'":
                if not in_string:
                    in_string = ch
                elif in_string == ch:
                    in_string = False
                continue
            if in_string:
                continue
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[start_idx:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        try:
                            cand2 = re.sub(r"'", '"', candidate)
                            cand2 = re.sub(r",\s*}", "}", cand2)
                            cand2 = re.sub(r",\s*]", "]", cand2)
                            return json.loads(cand2)
                        except Exception:
                            return None
    return None



def call_gemma_extract_rag(retrieved_chunks, field_schema, model=LM_MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    safe_chunks = []
    for r in (retrieved_chunks or []):
        if not isinstance(r, dict):
            continue
        text = r.get("text") or r.get("text_preview") or ""
        preview = r.get("text_preview") or (text[:200] if text else "")
        source = r.get("source") or r.get("doc_id") or "unknown"
        safe_chunks.append({"source": source, "text_preview": preview, "text": text})

    fields = field_schema if isinstance(field_schema, (list, tuple)) else list(field_schema.keys())
    schema_block = {
        "fields": [
            {"name": f, "description": (field_schema[f] if isinstance(field_schema, dict) else "")}
            for f in fields
        ],
        "rules": [
            "Output EXACTLY one valid JSON object and nothing else.",
            "Use null for missing fields.",
            "Dates use ISO 8601 if possible (YYYY-MM-DD or YYYY-MM-DDTHH:MM:SS).",
            "Do not hallucinate — if uncertain, set null and add a short note in `notes` field.",
        ]
    }

    header = "You are a data extraction engine. Given the retrieved document snippets below, extract the specified fields EXACTLY as a JSON object. Output JSON only.\n\n"
    header += json.dumps(schema_block, indent=2) + "\n\n"

    body = "## Retrieved snippets (most relevant first):\n\n"
    for i, r in enumerate(safe_chunks):
        body += f"--- snippet {i+1} (source: {r.get('source')}, preview: {r.get('text_preview')[:120]}) ---\n{r.get('text')}\n\n"

    field_example = ", ".join([f'"{f}": null' for f in fields])
    footer = f"\n\nOutput JSON with these keys: {{{field_example}, \"notes\": null}}\n\nDO NOT output any explanatory text.\n"

    prompt = header + body + footer

    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role":"user","content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature
        )
    except Exception as e:
        return None, f"LM call failed: {e}"

    try:
        assistant_text = resp.choices[0].message.content
    except Exception:
        try:
            assistant_text = resp.choices[0]['message']['content']
        except Exception:
            assistant_text = str(resp)

    parsed = extract_json_block(assistant_text)
    return parsed, assistant_text

In [15]:
# -------------------- Retrieval + LM (Gemma) --------------------
def load_vectorstore(index_path=VECTOR_INDEX_FILE, metadata_path=METADATA_FILE):
    if not os.path.exists(index_path) or not os.path.exists(metadata_path):
        raise FileNotFoundError("Index or metadata file not found. Run ingest first.")
    index = load_index(index_path)
    with open(metadata_path, "r", encoding="utf-8") as fh:
        metadata = json.load(fh)
    for m in metadata:
        if 'text' not in m:
            m['text'] = m.get('text_preview', '')
    return index, metadata

def retrieve(query, index, metadata, top_k=TOP_K):
    q_emb = embed_texts([query])[0:1]
    D, I = index.search(q_emb, top_k)
    results = []
    for idx in I[0]:
        if idx < 0 or idx >= len(metadata):
            continue
        results.append(metadata[idx])
    return results

def extract_json_block(text):
    if not text or "{" not in text:
        return None
    start_idx = None
    depth = 0
    in_string = False
    escape = False
    for i, ch in enumerate(text):
        if start_idx is None:
            if ch == "{":
                start_idx = i
                depth = 1
                in_string = False
                escape = False
            else:
                continue
        else:
            if escape:
                escape = False
                continue
            if ch == "\\":
                escape = True
                continue
            if ch == '"' or ch == "'":
                if not in_string:
                    in_string = ch
                elif in_string == ch:
                    in_string = False
                continue
            if in_string:
                continue
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[start_idx:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        try:
                            cand2 = re.sub(r"'", '"', candidate)
                            cand2 = re.sub(r",\s*}", "}", cand2)
                            cand2 = re.sub(r",\s*]", "]", cand2)
                            return json.loads(cand2)
                        except Exception:
                            return None
    return None

def call_gemma_extract_rag(retrieved_chunks, field_schema, model=LM_MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    # Defensive normalization of chunks
    safe_chunks = []
    for r in (retrieved_chunks or []):
        if not isinstance(r, dict):
            continue
        text = r.get("text") or r.get("text_preview") or ""
        preview = r.get("text_preview") or (text[:200] if text else "")
        source = r.get("source") or r.get("doc_id") or "unknown"
        safe_chunks.append({"source": source, "text_preview": preview, "text": text})

    fields = field_schema if isinstance(field_schema, (list, tuple)) else list(field_schema.keys())
    schema_block = {
        "fields": [
            {"name": f, "description": (field_schema[f] if isinstance(field_schema, dict) else "")}
            for f in fields
        ],
        "rules": [
            "Output EXACTLY one valid JSON object and nothing else.",
            "Use null for missing fields.",
            "Dates use ISO 8601 if possible (YYYY-MM-DD or YYYY-MM-DDTHH:MM:SS).",
            "Do not hallucinate — if uncertain, set null and add a short note in `notes` field.",
        ]
    }

    header = "You are a data extraction engine. Given the retrieved document snippets below, extract the specified fields EXACTLY as a JSON object. Output JSON only.\n\n"
    header += json.dumps(schema_block, indent=2) + "\n\n"
    body = "## Retrieved snippets (most relevant first):\n\n"
    for i, r in enumerate(safe_chunks):
        body += f"--- snippet {i+1} (source: {r.get('source')}, preview: {r.get('text_preview')[:120]}) ---\n{r.get('text')}\n\n"
    field_example = ", ".join([f'"{f}": null' for f in fields])
    footer = f"\n\nOutput JSON with these keys: {{{field_example}, \"notes\": null}}\n\nDO NOT output any explanatory text.\n"
    prompt = header + body + footer

    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role":"user","content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature
        )
    except Exception as e:
        return None, f"LM call failed: {e}"

    try:
        assistant_text = resp.choices[0].message.content
    except Exception:
        try:
            assistant_text = resp.choices[0]['message']['content']
        except Exception:
            assistant_text = str(resp)

    parsed = extract_json_block(assistant_text)
    return parsed, assistant_text

        

In [ ]:
# -------------------- FIELD SCHEMA + extractor --------------------
FIELD_SCHEMA = {
    "nama_pemohon": "Nama pemohon yang submit permohonan lelang",
    "transaction_date": "Tanggal terjadinya transaksi atau dokumen dibuat (ISO 8601 preferred)",
    "subjek_pemohon": "Subjek hukum yang membuat permohonan lelang",
    "tanggal_mulai_lelang": "Tanggal Lelang Dimulai (Timestamped)",
    "tanggal_selesai_lelang": "Tanggal Lelang Berakhir (Timestamped)",
    "kpknl": "KPKNL yang melaksanakan lelang",
    "harga_laku_lelang": "Nilai transaksi yang dibid oleh pemenang lelang",
    "id_dokumen": "Identifier dokumen terkait (source path or assigned id)",
    "catatan": "Catatan atau ketidakpastian"
}

def extract_from_all_documents(index=None, metadata=None, index_path=VECTOR_INDEX_FILE, metadata_path=METADATA_FILE, output_csv=OUTPUT_CSV, top_k=TOP_K, debug=False):
    # load or use in-memory
    if index is None or metadata is None:
        index, metadata = load_vectorstore(index_path, metadata_path)

    for m in metadata:
        if 'text' not in m:
            m['text'] = m.get('text_preview', '')

    docs = {}
    for m in metadata:
        docs.setdefault(m['doc_id'], []).append(m)

    results = []
    debug_outputs = []

    for doc_i, (doc_id, chunks) in enumerate(docs.items(), start=1):
        print(f"[{doc_i}/{len(docs)}] Processing doc_id={doc_id}, source={chunks[0].get('source')}")
        combined_text = " ".join([c.get('text','') for c in chunks])
        retrieved_chunks = retrieve(combined_text, index, metadata, top_k=top_k)
        parsed_json, raw_text = call_gemma_extract_rag(retrieved_chunks, FIELD_SCHEMA)

        if parsed_json is None:
            print(f"  [WARN] No valid JSON returned for {doc_id}. Saving debug raw output.")
            parsed_json = {k: None for k in list(FIELD_SCHEMA.keys())}
            parsed_json['notes'] = "model_no_json"
            debug_outputs.append({"doc_id": doc_id, "source": chunks[0].get('source'), "raw_text": raw_text, "retrieved_preview": [r.get('text_preview') for r in retrieved_chunks]})
        else:
            parsed_json.setdefault('notes', None)

        parsed_json['id_dokumen'] = chunks[0].get('source')
        results.append(parsed_json)
        time.sleep(0.15)

    # write CSV
    fieldnames = list(FIELD_SCHEMA.keys()) + ["id_dokumen", "notes"]
    with open(output_csv, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        for r in results:
            row = {k: r.get(k, None) for k in fieldnames}
            writer.writerow(row)
    print(f"Saved extracted results to {output_csv}")

    if debug and debug_outputs:
        with open("debug_raw_outputs.json", "w", encoding="utf-8") as fh:
            json.dump(debug_outputs, fh, indent=2, ensure_ascii=False)
        print(f"Saved debug details to debug_raw_outputs.json")

    return results



In [ ]:
### BATASAN MODEL BARU 3.x 

In [34]:
# Tesseract path (Windows) - set if needed (adjust to your installation)
# Example: r"C:\Program Files\Tesseract-OCR\tesseract.exe"
TESSERACT_CMD = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # <-- edit or set to None to skip
if TESSERACT_CMD:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

# Embeddings model
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM = 384

# RAG / chunking config
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 6

# Default file names (used if not using file-specific naming)
VECTOR_INDEX_FILE = "rag_index.faiss"
METADATA_FILE = "rag_metadata.json"
OUTPUT_CSV = "extracted_dok_lelang.csv"

# Other
DEBUG_OCR_DIR = "ocr_debug"
TEMPERATURE = 0.0
MAX_TOKENS = 1024
# -----------------------------------------------



In [35]:
# LM client
client = OpenAI(base_url=LM_BASE_URL, api_key=LM_API_KEY)



In [36]:
# -------------------- UTIL: filenames --------------------
def sanitize_basename(path):
    p = Path(path)
    base = p.stem.lower()
    base = re.sub(r'[^a-z0-9_]+', '_', base)
    base = re.sub(r'_{2,}', '_', base).strip('_')
    return base or "doc"




In [37]:
# -------------------- OCR helpers --------------------
def load_image_cv2(path):
    """Robust image load: supports unicode path on Windows."""
    arr = np.fromfile(path, dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None:
        # fallback via PIL
        img = cv2.cvtColor(np.array(Image.open(path)), cv2.COLOR_RGB2BGR)
    return img

def resize_max(img, max_side=2400, min_side=800):
    h, w = img.shape[:2]
    max_current = max(h, w)
    if max_current > max_side:
        scale = max_side / max_current
        return cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_CUBIC)
    if max_current < min_side:
        scale = min_side / max_current
        return cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_CUBIC)
    return img

def to_grayscale(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def denoise(img_gray):
    return cv2.fastNlMeansDenoising(img_gray, h=10)

def adaptive_thresh(img_gray):
    return cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)

def otsu_thresh(img_gray):
    _, th = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return th

def unsharp_mask(img_gray):
    blur = cv2.GaussianBlur(img_gray, (0,0), sigmaX=3)
    return cv2.addWeighted(img_gray, 1.5, blur, -0.5, 0)

def deskew(img_gray):
    coords = np.column_stack(np.where(img_gray < 255))
    if coords.size == 0:
        return img_gray
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    (h, w) = img_gray.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(img_gray, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return rotated

def ocr_with_confidence(pil_img, config="--oem 3 --psm 3"):
    """
    Returns (text, avg_conf, total_chars). Uses image_to_data for per-word confidence.
    """
    try:
        data = pytesseract.image_to_data(pil_img, config=config, output_type=pytesseract.Output.DICT)
    except Exception as e:
        return "", 0.0, 0
    texts = []
    confs = []
    for txt, conf in zip(data.get('text', []), data.get('conf', [])):
        if txt and str(txt).strip():
            # conf can be '-1' or non-int
            try:
                ic = int(float(conf))
            except Exception:
                ic = 0
            if ic > 0:
                texts.append(txt)
                confs.append(ic)
            else:
                # still keep short words if no confidences
                texts.append(txt)
    joined = " ".join([t for t in texts]).strip()
    avg_conf = (sum(confs) / len(confs)) if confs else 0.0
    total_chars = len(joined)
    return joined, avg_conf, total_chars

def try_preprocess_and_ocr(path, debug_dir=DEBUG_OCR_DIR):
    """
    Try multiple preprocessing variants and pytesseract configs.
    Save debug images + candidate JSON to debug_dir.
    Returns best_text, best_candidate_dict
    """
    os.makedirs(debug_dir, exist_ok=True)
    img = load_image_cv2(path)
    img = resize_max(img, max_side=2600, min_side=900)
    gray = to_grayscale(img)

    variants = []
    variants.append(("gray", gray))
    try:
        variants.append(("deskew", deskew(gray)))
    except Exception:
        pass
    try:
        variants.append(("denoise", denoise(gray)))
    except Exception:
        pass
    try:
        variants.append(("otsu", otsu_thresh(gray)))
    except Exception:
        pass
    try:
        variants.append(("adaptive", adaptive_thresh(gray)))
    except Exception:
        pass
    try:
        variants.append(("unsharp", unsharp_mask(gray)))
    except Exception:
        pass

    morphs = []
    for lab, v in variants:
        morphs.append((lab + "_none", v))
        try:
            k = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
            morphs.append((lab + "_open", cv2.morphologyEx(v, cv2.MORPH_OPEN, k)))
            morphs.append((lab + "_close", cv2.morphologyEx(v, cv2.MORPH_CLOSE, k)))
        except Exception:
            pass

    configs = ["--oem 3 --psm 6", "--oem 3 --psm 3", "--oem 3 --psm 11", "--oem 1 --psm 6"]
    candidates = []
    for lab, proc in morphs:
        try:
            pil_img = Image.fromarray(proc) if len(proc.shape) == 2 else Image.fromarray(cv2.cvtColor(proc, cv2.COLOR_BGR2RGB))
        except Exception:
            pil_img = Image.fromarray(cv2.cvtColor(proc, cv2.COLOR_BGR2RGB)) if len(proc.shape) != 2 else Image.fromarray(proc)
        for cfg in configs:
            text, avg_conf, total_chars = ocr_with_confidence(pil_img, config=cfg)
            candidates.append({
                "variant": lab,
                "config": cfg,
                "text": text,
                "avg_conf": avg_conf,
                "total_chars": total_chars
            })

    # pick best: highest total_chars first, tie-break by avg_conf
    best = max(candidates, key=lambda c: (c["total_chars"], c["avg_conf"]))
    # save debug images + candidates
    try:
        # save best image
        for lab, proc in morphs:
            if lab == best["variant"]:
                out_img = os.path.join(debug_dir, f"debug_best_{Path(path).stem}.png")
                cv2.imwrite(out_img, proc if proc.dtype == np.uint8 else proc.astype(np.uint8))
                break
        with open(os.path.join(debug_dir, f"debug_candidates_{Path(path).stem}.json"), "w", encoding="utf-8") as fh:
            json.dump(candidates, fh, indent=2, ensure_ascii=False)
    except Exception:
        pass

    return best["text"], best



In [38]:
# -------------------- PDF helpers --------------------
def pdf_to_text(path, ocr_images=False, lang='ind+eng'):
    """Extract text from PDF, with optional per-page OCR fallback."""
    text_pages = []
    try:
        with pdfplumber.open(path) as pdf:
            for p in pdf.pages:
                page_text = p.extract_text()
                if page_text and page_text.strip():
                    text_pages.append(page_text)
                else:
                    if ocr_images:
                        pil = p.to_image(resolution=300).original
                        page_text = pytesseract.image_to_string(pil, lang=lang)
                        text_pages.append(page_text)
    except Exception as e:
        # fallback: try external preprocessing OCR of whole PDF as image(s)
        try:
            text, best = try_preprocess_and_ocr(path, debug_dir=DEBUG_OCR_DIR)
            return text
        except Exception:
            return ""
    return "\n\n".join(text_pages)

def image_to_text(path, lang='ind+eng'):
    """Simple direct OCR via pytesseract; returns empty string if fails."""
    try:
        # load via pillow (handles unicode paths)
        text = pytesseract.image_to_string(Image.open(path), lang=lang)
        return text
    except Exception:
        return ""



In [39]:
# -------------------- TEXT cleaning + chunking --------------------
def clean_text(text):
    if not text:
        return ""
    t = re.sub(r'\r\n', '\n', text)
    t = re.sub(r'\n{3,}', '\n\n', t)
    t = re.sub(r'[ \t]+', ' ', t)
    return t.strip()

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = clean_text(text)
    if not text:
        return []
    n = len(text)
    chunks = []
    start = 0
    cid = 0
    while start < n:
        end = start + chunk_size
        chunk_txt = text[start:end]
        chunks.append({"id": cid, "text": chunk_txt, "start": start, "end": min(end, n)})
        cid += 1
        if end >= n:
            break
        start = end - overlap
    return chunks




In [40]:
# -------------------- Embedding + FAISS helpers --------------------
print("Loading embedding model:", EMBED_MODEL_NAME)
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

def create_faiss_index(dim=EMBED_DIM):
    return faiss.IndexFlatIP(dim)

def save_index(index, filepath):
    faiss.write_index(index, filepath)

def load_index(filepath):
    return faiss.read_index(filepath)

def embed_texts(texts, batch_size=32):
    embs = embed_model.encode(texts, convert_to_numpy=True, show_progress_bar=False, batch_size=batch_size)
    faiss.normalize_L2(embs)
    return embs



        

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


In [41]:
# -------------------- Ingest (single file/folder) --------------------
def ingest_single_file(path, index_path=None, metadata_path=None, ocr_for_pdf=True):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"{path} not found")

    base = sanitize_basename(path)
    if index_path is None:
        index_path = f"rag_index_{base}.faiss"
    if metadata_path is None:
        metadata_path = f"rag_meta_{base}.json"

    ext = p.suffix.lower()
    if ext == ".pdf":
        raw = pdf_to_text(str(p), ocr_images=ocr_for_pdf)
    elif ext in [".png", ".jpg", ".jpeg", ".tif", ".tiff"]:
        # first try direct OCR
        raw = image_to_text(str(p))
        # fallback to preprocessing if direct OCR poor
        if not raw or len(raw.strip()) < 20:
            print("[WARN] Direct image OCR returned little/no text — trying preprocessing OCR...")
            raw, best = try_preprocess_and_ocr(str(p), debug_dir=DEBUG_OCR_DIR)
    elif ext == ".txt":
        raw = p.read_text(encoding="utf-8", errors="ignore")
    else:
        raise ValueError("Unsupported file type")

    raw = clean_text(raw)
    if not raw:
        raise RuntimeError("No text extracted. Try ocr_for_pdf=True for scanned PDFs or check ocr_debug/ for debug images.")

    chunks = chunk_text(raw)
    if not chunks:
        raise RuntimeError("Chunking produced zero chunks (empty text after cleaning).")
    all_chunks = []
    metadata = []
    doc_id = base
    for c in chunks:
        uid = f"{doc_id}_c{c['id']}"
        all_chunks.append((uid, c["text"]))
        metadata.append({
            "uid": uid,
            "doc_id": doc_id,
            "source": str(p),
            "start": c["start"],
            "end": c["end"],
            "text": c["text"],
            "text_preview": c["text"][:400]
        })

    texts = [t for (_, t) in all_chunks]
    embs = embed_texts(texts)
    index = create_faiss_index(dim=embs.shape[1])
    index.add(embs)

    save_index(index, index_path)
    with open(metadata_path, "w", encoding="utf-8") as fh:
        json.dump(metadata, fh, indent=2, ensure_ascii=False)

    print(f"Ingested {len(all_chunks)} chunks from {path}")
    print(f"Saved index -> {index_path}")
    print(f"Saved metadata -> {metadata_path}")
    return index, metadata, index_path, metadata_path

def ingest_folder(folder, index_path=VECTOR_INDEX_FILE, metadata_path=METADATA_FILE, ocr_for_pdf=True):
    patterns = [os.path.join(folder, ext) for ext in ["**/*.pdf", "**/*.png", "**/*.jpg", "**/*.jpeg", "**/*.tif", "**/*.tiff", "**/*.txt"]]
    files = []
    for pat in patterns:
        files.extend(glob.glob(pat, recursive=True))

    all_chunks = []
    metadata = []
    doc_counter = 0
    for path in tqdm(sorted(set(files)), desc="Files"):
        ext = os.path.splitext(path)[1].lower()
        if ext == ".pdf":
            raw = pdf_to_text(path, ocr_images=ocr_for_pdf)
        elif ext in [".png", ".jpg", ".jpeg", ".tif", ".tiff"]:
            raw = image_to_text(path)
            if not raw or len(raw.strip()) < 20:
                raw, best = try_preprocess_and_ocr(path, debug_dir=DEBUG_OCR_DIR)
        elif ext == ".txt":
            with open(path, "r", encoding="utf-8", errors="ignore") as fh:
                raw = fh.read()
        else:
            raw = ""
        raw = clean_text(raw)
        if not raw:
            continue
        doc_id = f"doc_{doc_counter}"
        doc_counter += 1
        chunks = chunk_text(raw)
        for c in chunks:
            uid = f"{doc_id}_c{c['id']}"
            all_chunks.append((uid, c["text"]))
            metadata.append({"uid": uid, "doc_id": doc_id, "source": path, "start": c["start"], "end": c["end"], "text": c["text"], "text_preview": c["text"][:200]})

    if not all_chunks:
        print("No text chunks to ingest.")
        return None, None

    texts = [t for (_, t) in all_chunks]
    embs = embed_texts(texts)
    index = create_faiss_index(dim=embs.shape[1])
    index.add(embs)

    with open(metadata_path, "w", encoding="utf-8") as fh:
        json.dump(metadata, fh, indent=2, ensure_ascii=False)
    save_index(index, index_path)
    print(f"Ingested {len(all_chunks)} chunks from {len(metadata)} source chunks. Index saved to {index_path}")
    return index, metadata

def ingest_path(path, index_dir=".", reuse=True, ocr_for_pdf=True):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"{path} not found")
    if p.is_file():
        base = sanitize_basename(path)
        index_path = os.path.join(index_dir, f"rag_index_{base}.faiss")
        metadata_path = os.path.join(index_dir, f"rag_meta_{base}.json")
        if reuse and os.path.exists(index_path) and os.path.exists(metadata_path):
            print(f"Reusing index/metadata for {path}")
            idx = load_index(index_path)
            with open(metadata_path, "r", encoding="utf-8") as fh:
                md = json.load(fh)
            return idx, md, index_path, metadata_path
        return ingest_single_file(path, index_path=index_path, metadata_path=metadata_path, ocr_for_pdf=ocr_for_pdf)
    else:
        # folder: return index, metadata plus their paths for consistency
        idx, md = ingest_folder(path, index_path=os.path.join(index_dir, VECTOR_INDEX_FILE), metadata_path=os.path.join(index_dir, METADATA_FILE), ocr_for_pdf=ocr_for_pdf)
        return idx, md, os.path.join(index_dir, VECTOR_INDEX_FILE), os.path.join(index_dir, METADATA_FILE)



In [45]:
# -------------------- USAGE example --------------------
if __name__ == "__main__":
    # adjust this to the file you want to test (png/jpg/pdf)
    test_file = Path("Permohonan lelang BMN PN Lahat.pdf")
    if not test_file.exists():
        print("Test file not found:", test_file)
    else:
        # ingest file (force fresh ingest)
        idx, md, idx_path, meta_path = ingest_path(str(test_file), index_dir=".", reuse=False, ocr_for_pdf=True)
        # extract results using in-memory index/metadata (guaranteed file-specific)
        res = extract_from_all_documents(index=idx, metadata=md, output_csv=f"extracted_{sanitize_basename(test_file)}.csv", top_k=8, debug=True)
        print("RESULTS PREVIEW:", res[:2])

        

RuntimeError: No text extracted. Try ocr_for_pdf=True for scanned PDFs or check ocr_debug/ for debug images.